# Qwen2.5-1.5B financial continued pretraining with QLoRA

A complete Google Colab workflow for **Qwen/Qwen2.5-1.5B (base)** and
**alwaysgood/financial-english-source-corpus**. This is next-token learning on
financial prose, not instruction tuning. Inference uses plain text completion.

**Start:** upload this notebook to [Google Colab](https://colab.research.google.com/),
choose **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU), then run
the cells in order. Allow Google Drive access when prompted. Use a fresh runtime;
if libraries were already imported, restart the session after the installation cell.
Public model/data downloads do not require a Hugging Face token. Optional authentication
uses a Colab secret named `HF_TOKEN`.

Defaults are a bounded **3,000-document streaming sample**, 1,024-token blocks,
and **100 optimizer steps**. This is a practical starter run, not a full-corpus run.
GPU time varies; there is no guaranteed improvement from such a short run. Training
and checkpoints require free Drive space and a working internet connection.

The Hub viewer inspected on 2026-11-06 reports a `train` split with columns
`source` (text), `source_tokens`, `dataset`, `metadata_json`, `source_file`, and `index`.
Runtime checks verify the actual schema. We retokenize with Qwen2.5; `source_tokens`
was produced with a different tokenizer and is not used for training.


## 1. Install a pinned, compatible training stack

These deliberately fixed versions match the TRL 0.17 API. The trainer cell also
checks argument names instead of mixing old `tokenizer`/`max_seq_length` examples
with newer `processing_class`/`max_length` APIs. Arbitrary future versions are not
guaranteed. Keep Colab's CUDA-enabled PyTorch; do not install a CPU PyTorch wheel.


In [ ]:
import subprocess, sys
PACKAGES = [
    "transformers==4.51.3", "trl==0.17.0", "peft==0.15.2",
    "accelerate==1.6.0", "datasets==3.5.0", "bitsandbytes==0.48.2",
    "huggingface_hub==0.30.2", "numpy<3",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PACKAGES])
print("Installation complete. If these libraries were already imported, restart the session now.")


Installation complete. If these libraries were already imported, restart the session now.


## 2. Configuration and GPU checks

Choose a CUDA GPU runtime: A100, L4, T4, or another supported NVIDIA GPU.
Native BF16 support selects BF16; otherwise FP16 is used with Trainer loss scaling.
MICRO_BATCH=None chooses a conservative memory-based default. Override it with a
positive divisor of EFFECTIVE_BATCH (default 8); GRAD_ACCUM is calculated automatically.
For OOM, reduce MICRO_BATCH and rebuild the model/trainer.
For a smoke run set MAX_STEPS=2, EVAL_BLOCKS=4, EVAL_EVERY=2, SAVE_EVERY=2.


In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
import gc, math, json, random, hashlib, inspect, itertools, unicodedata, subprocess, sys
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTConfig, SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-1.5B"
DATASET_ID = "alwaysgood/financial-english-source-corpus"
MODEL_REVISION = "main"  # Resolved to an immutable SHA below; paste a SHA to reproduce.
DATASET_REVISION = "main"
DATASET_CONFIG = None
DATASET_SPLIT = "train"
TEXT_FIELD = "source"   # None allows unambiguous inference from source/text/content/document.
SEED = 42
SAMPLE_DOCS = 3000
MAX_SCANNED_ROWS = 30000
SHUFFLE_BUFFER = 1000
MIN_CHARS = 80
MAX_DOC_CHARS = 32768  # Explicit memory bound; long documents are truncated and counted.
BLOCK_SIZE = 1024
EVAL_FRACTION = 0.05
EVAL_BLOCKS = 64
MAX_STEPS = 100
MICRO_BATCH = None
EFFECTIVE_BATCH = 8
DEFAULT_LARGE_GPU_BATCH = 1
LEARNING_RATE = 1e-4
EVAL_EVERY = 50
SAVE_EVERY = 50
LORA_R = 16
LORA_ALPHA = 32
DRIVE_ROOT = "/content/drive/MyDrive/qwen_financial_qlora"
RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_%f")

if not torch.cuda.is_available():
    raise RuntimeError("Select an NVIDIA GPU in Runtime → Change runtime type, then rerun.")
assert 128 <= BLOCK_SIZE <= 4096
assert SAMPLE_DOCS >= 100 and MAX_SCANNED_ROWS >= SAMPLE_DOCS
assert 0 < EVAL_FRACTION < 0.5 and EVAL_BLOCKS >= 1
assert MAX_STEPS > 0 and EVAL_EVERY > 0 and SAVE_EVERY > 0
assert SHUFFLE_BUFFER > 0 and MAX_DOC_CHARS >= MIN_CHARS > 0
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEMORY_GIB = torch.cuda.get_device_properties(0).total_memory / 2**30
BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
if MICRO_BATCH is None:
    MICRO_BATCH = DEFAULT_LARGE_GPU_BATCH if BF16 and GPU_MEMORY_GIB >= 35 else 1
if not isinstance(MICRO_BATCH, int) or MICRO_BATCH <= 0:
    raise ValueError("MICRO_BATCH must be a positive integer or None.")
if EFFECTIVE_BATCH <= 0 or EFFECTIVE_BATCH % MICRO_BATCH:
    raise ValueError("EFFECTIVE_BATCH must be positive and divisible by MICRO_BATCH.")
GRAD_ACCUM = EFFECTIVE_BATCH // MICRO_BATCH
set_seed(SEED)
versions = {p: version(p) for p in ["torch", "transformers", "trl", "peft", "datasets", "accelerate", "bitsandbytes", "huggingface_hub"]}
print(versions)
print("Microbatch:", MICRO_BATCH, "| Gradient accumulation:", GRAD_ACCUM)
print("GPU:", torch.cuda.get_device_name(0), "| compute:", COMPUTE_DTYPE)
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Effective sequences/step:", MICRO_BATCH * GRAD_ACCUM, "| tokens/step:", MICRO_BATCH * GRAD_ACCUM * BLOCK_SIZE)
if version("transformers") != "4.51.3" or version("trl") != "0.17.0":
    raise RuntimeError("Pinned libraries are not active. Restart the session after installation.")


{'torch': '2.14.1+cu130', 'transformers': '4.51.3', 'trl': '0.17.0', 'peft': '0.15.2', 'datasets': '3.5.0', 'accelerate': '1.6.0', 'bitsandbytes': '0.48.2', 'huggingface_hub': '0.30.2'}
GPU: Tesla T4 | compute: torch.bfloat16
GPU memory (GiB): 14.6
Effective sequences/step: 8 | tokens/step: 8192


## 3. Mount Drive and resolve immutable Hub revisions

Each run uses a new directory. Checkpoints, the final LoRA adapter, tokenizer,
metrics, configuration, and document-hash split manifest are saved there.
The final adapter needs the original base model; it is not a standalone full model.


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None  # A missing/denied optional secret is fine for these public repos.
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
del hf_token

api = HfApi()
MODEL_SHA = api.model_info(MODEL_ID, revision=MODEL_REVISION).sha
DATASET_SHA = api.dataset_info(DATASET_ID, revision=DATASET_REVISION).sha
RUN_DIR = Path(DRIVE_ROOT) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
ADAPTER_DIR = RUN_DIR / "adapter"
def save_json(path, obj):
    Path(path).write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False))
config = {k: v for k, v in list(globals().items()) if k.isupper() and isinstance(v, (str, int, float, bool, type(None)))}
config.update(model_sha=MODEL_SHA, dataset_sha=DATASET_SHA, versions=versions,
              run_dir=str(RUN_DIR), adapter_dir=str(ADAPTER_DIR))
save_json(RUN_DIR / "run_config.json", config)
print("Run folder:", RUN_DIR)
print("Model SHA:", MODEL_SHA, "| Dataset SHA:", DATASET_SHA)


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Run folder: /content/drive/MyDrive/qwen_financial_qlora/run_20261106_220952_568494
Model SHA: 8faed761d45a263340a0528343f099c05c9a4323 | Dataset SHA: cb43595ddec84b81a07977ab2f7c63a5547971c0


## 4. Inspect schema and collect a bounded streaming sample

This streams Parquet instead of downloading the full corpus into RAM. Buffered
shuffle mixes shards/nearby rows but is **not a globally uniform random sample**;
the source-label histogram below helps reveal bias. Increase the buffer/sample for
broader coverage. Only the selected text column becomes training input. Metadata,
URLs, indices, and precomputed token counts are not appended to the text.

Exact duplicate detection uses Unicode-normalized, whitespace-normalized text
after the character cap. This does not guarantee removal of near duplicates or
related excerpts from the same filing; held-out scores are a local diagnostic.


In [ ]:
stream = load_dataset(
    DATASET_ID, name=DATASET_CONFIG, split=DATASET_SPLIT,
    revision=DATASET_SHA, streaming=True,
)
print("Features:", stream.features)
preview = list(itertools.islice(iter(stream), 5))
if not preview:
    raise ValueError("The selected dataset split is empty.")
print("Columns:", list(preview[0]))
print("First-row types:", {k: type(v).__name__ for k, v in preview[0].items()})

def select_text_field(rows, requested):
    columns = set(rows[0])
    if requested is not None:
        if requested not in columns:
            raise ValueError(f"TEXT_FIELD={requested!r} is absent. Columns: {sorted(columns)}")
        if not any(isinstance(r.get(requested), str) and r[requested].strip() for r in rows):
            raise TypeError(f"{requested!r} has no nonempty string in the preview. Inspect schema before training.")
        return requested
    choices = [k for k in ("source", "text", "content", "document")
               if k in columns and any(isinstance(r.get(k), str) and r[k].strip() for r in rows)]
    if len(choices) != 1:
        raise ValueError(f"Ambiguous/missing text field: {choices}. Set TEXT_FIELD explicitly.")
    return choices[0]

def clean_text(value):
    if not isinstance(value, str):
        return None, False
    value = unicodedata.normalize("NFKC", value).replace("\x00", "").strip()
    capped = len(value) > MAX_DOC_CHARS
    value = value[:MAX_DOC_CHARS].strip()
    if len(value) < MIN_CHARS:
        return None, capped
    return value, capped

def text_hash(text):
    return hashlib.sha256(" ".join(text.split()).encode("utf-8")).hexdigest()

selected_field = select_text_field(preview, TEXT_FIELD)
print("Training text field:", selected_field)
preview_text = next(r[selected_field] for r in preview if isinstance(r.get(selected_field), str) and r[selected_field].strip())
print("Preview (first 300 characters):", repr(preview_text[:300]))
docs, seen = [], set()
sample_stats = Counter()
shuffled = stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)
for row in itertools.islice(shuffled, MAX_SCANNED_ROWS):
    sample_stats["scanned"] += 1
    if selected_field not in row:
        raise ValueError("Inconsistent schema: selected text field missing in a streamed row.")
    text, capped = clean_text(row[selected_field])
    sample_stats["character_capped"] += int(capped)
    if text is None:
        sample_stats["invalid_or_short"] += 1
        continue
    digest = text_hash(text)
    if digest in seen:
        sample_stats["duplicates"] += 1
        continue
    seen.add(digest)
    docs.append({"text": text, "sha256": digest, "source_label": str(row.get("dataset", "unknown"))})
    if len(docs) == SAMPLE_DOCS:
        break
if len(docs) < 100:
    raise ValueError(f"Only {len(docs)} usable documents. Inspect the schema or raise MAX_SCANNED_ROWS.")
if len(docs) < SAMPLE_DOCS:
    print(f"Warning: scan ended with {len(docs)} of {SAMPLE_DOCS} requested documents.")
print("Sampling:", dict(sample_stats), "| accepted:", len(docs))
print("Source labels:", dict(Counter(d["source_label"] for d in docs)))


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/23 [00:00<?, ?it/s]

Features: {'source': Value(dtype='string', id=None), 'source_tokens': Value(dtype='int32', id=None), 'dataset': Value(dtype='string', id=None), 'metadata_json': Value(dtype='string', id=None), 'source_file': Value(dtype='string', id=None), 'index': Value(dtype='int64', id=None)}
Columns: ['source', 'source_tokens', 'dataset', 'metadata_json', 'source_file', 'index']
First-row types: {'source': 'str', 'source_tokens': 'int', 'dataset': 'str', 'metadata_json': 'str', 'source_file': 'str', 'index': 'int'}
Training text field: source
Preview (first 300 characters): 'Export taxes on cocoa beans from Ivory Coast, the world’s biggest producer of the chocolate ingredient, won’t exceed 22 percent of the international price this season, meeting a commitment to the International Monetary Fund, according to a finance ministry document. In the 2008-9 season taxes averag'
Sampling: {'scanned': 3336, 'character_capped': 0, 'invalid_or_short': 336} | accepted: 3000
Source labels: {'10k': 3000}


## 5. Split documents, tokenize, and pack each split independently

The split happens **before** tokenization/packing, after exact deduplication.
Every document receives its tokenizer EOS token. Fixed-length blocks concatenate
documents within one split, so attention can cross EOS boundaries (standard
causal pretraining). Each split drops at most `BLOCK_SIZE - 1` trailing tokens.
Evaluation uses a fixed shuffled subset of at most `EVAL_BLOCKS` blocks.
No chat template or assistant-only loss mask is applied.


In [ ]:
ordered_docs = list(docs)
random.Random(SEED).shuffle(ordered_docs)
n_eval = max(16, round(len(ordered_docs) * EVAL_FRACTION))
eval_docs, train_docs = ordered_docs[:n_eval], ordered_docs[n_eval:]
assert set(d["sha256"] for d in train_docs).isdisjoint(d["sha256"] for d in eval_docs)
split_manifest = {"train": [{k: d[k] for k in ("sha256", "source_label")} for d in train_docs],
                  "eval": [{k: d[k] for k in ("sha256", "source_label")} for d in eval_docs]}
save_json(RUN_DIR / "split_manifest.json", split_manifest)
print("Documents:", len(train_docs), "train /", len(eval_docs), "eval")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_SHA, use_fast=True)
if tokenizer.eos_token_id is None:
    raise ValueError("A valid EOS token is required for document boundaries.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def pack_documents(documents, tok, block_size):
    blocks, pending = [], []
    tokens = 0
    for doc in documents:
        ids = tok(doc["text"], add_special_tokens=False, truncation=False, verbose=False)["input_ids"]
        ids = ids + [tok.eos_token_id]
        tokens += len(ids)
        pending.extend(ids)
        offset = 0
        while len(pending) - offset >= block_size:
            blocks.append(pending[offset:offset + block_size])
            offset += block_size
        pending = pending[offset:]
    if not blocks:
        raise ValueError("Not enough tokens for one block. Increase the sample or lower BLOCK_SIZE.")
    return Dataset.from_dict({"input_ids": blocks}), {"tokens": tokens, "blocks": len(blocks), "tail_tokens_dropped": len(pending)}

train_data, train_token_stats = pack_documents(train_docs, tokenizer, BLOCK_SIZE)
eval_data, eval_token_stats = pack_documents(eval_docs, tokenizer, BLOCK_SIZE)
eval_data = eval_data.shuffle(seed=SEED).select(range(min(EVAL_BLOCKS, len(eval_data))))
print("Train:", train_token_stats, "| Eval:", eval_token_stats)
print("Evaluated blocks:", len(eval_data), "| Approximate training passes:", round(MAX_STEPS * MICRO_BATCH * GRAD_ACCUM / len(train_data), 2))
data_stats = {"sampling": dict(sample_stats), "train": train_token_stats, "eval": eval_token_stats,
              "eval_blocks_used": len(eval_data), "text_field": selected_field}
save_json(RUN_DIR / "data_stats.json", data_stats)
# Keep raw corpus text out of saved artifacts; hashes identify the split.
del docs, ordered_docs, train_docs, eval_docs, preview, stream, shuffled, seen
gc.collect()


Documents: 2850 train / 150 eval


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Train: {'tokens': 107674, 'blocks': 105, 'tail_tokens_dropped': 154} | Eval: {'tokens': 5582, 'blocks': 5, 'tail_tokens_dropped': 462}
Evaluated blocks: 5 | Approximate training passes: 7.62


191

## 6. Load the frozen NF4 base and attach trainable LoRA adapters

NF4, double quantization, gradient checkpointing, and a single-GPU device map
keep memory down. T4 uses float16 compute; capable GPUs use bfloat16. LoRA covers
Qwen attention and MLP projection layers. Only the adapter parameters are trained.


In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_SHA, quantization_config=quant_config,
    torch_dtype=COMPUTE_DTYPE, device_map={"": 0}, attn_implementation="sdpa",
)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
missing = [name for name in targets if not any(n.endswith("." + name) for n, _ in model.named_modules())]
if missing:
    raise ValueError(f"Qwen projection modules missing: {missing}")
lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules=targets,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
assert model.is_loaded_in_4bit
assert any(p.requires_grad for p in model.parameters())
assert all("lora_" in name for name, p in model.named_parameters() if p.requires_grad)


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


## 7. Build a version-aware TRL trainer and check one batch

The collator copies token IDs to labels; the model shifts them internally for
next-token prediction. Real EOS tokens remain supervised even when pad equals
EOS. These blocks have no padding. The general padding path masks positions by
attention mask, not token value. Evaluation returns loss only to avoid storing
vocabulary-sized logits. Training/evaluation logs stay local and in Drive.


In [ ]:
class CausalLMCollator:
    def __init__(self, tok):
        self.tok = tok
    def __call__(self, examples):
        batch = self.tok.pad(
            [{"input_ids": x["input_ids"], "attention_mask": [1] * len(x["input_ids"])} for x in examples],
            padding=True, return_tensors="pt",
        )
        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels
        return batch

collator = CausalLMCollator(tokenizer)
test_batch = collator([train_data[0]])
assert test_batch["input_ids"].shape == (1, BLOCK_SIZE)
assert torch.equal(test_batch["labels"], test_batch["input_ids"])
del test_batch

cfg_fields = inspect.signature(SFTConfig).parameters
trainer_fields = inspect.signature(SFTTrainer.__init__).parameters
cfg_kwargs = dict(
    output_dir=str(CHECKPOINT_DIR), max_steps=MAX_STEPS,
    per_device_train_batch_size=MICRO_BATCH, per_device_eval_batch_size=MICRO_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE, lr_scheduler_type="cosine", warmup_ratio=0.05,
    weight_decay=0.01, max_grad_norm=1.0,
    bf16=BF16, fp16=not BF16,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit", logging_steps=5, logging_first_step=True,
    eval_steps=EVAL_EVERY, save_strategy="steps", save_steps=SAVE_EVERY,
    save_total_limit=2, load_best_model_at_end=False,
    prediction_loss_only=True, report_to="none", push_to_hub=False,
    seed=SEED, data_seed=SEED, dataloader_num_workers=0,
    remove_unused_columns=False, label_names=["labels"], packing=False,
)
strategy_key = "eval_strategy" if "eval_strategy" in cfg_fields else "evaluation_strategy"
cfg_kwargs[strategy_key] = "steps"
length_key = "max_length" if "max_length" in cfg_fields else "max_seq_length"
if length_key not in cfg_fields or "dataset_kwargs" not in cfg_fields:
    raise RuntimeError("Unsupported TRL API. Reinstall the pinned stack and restart the session.")
cfg_kwargs[length_key] = BLOCK_SIZE
cfg_kwargs["dataset_kwargs"] = {"skip_prepare_dataset": True}
for name in ("completion_only_loss", "assistant_only_loss", "padding_free"):
    if name in cfg_fields:
        cfg_kwargs[name] = False
unsupported = set(cfg_kwargs) - set(cfg_fields)
if unsupported:
    raise RuntimeError(f"Unsupported training arguments {unsupported}; restore the pinned stack.")
training_args = SFTConfig(**cfg_kwargs)
trainer_kwargs = dict(model=model, args=training_args, train_dataset=train_data,
                      eval_dataset=eval_data, data_collator=collator)
if "processing_class" in trainer_fields:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_fields:
    trainer_kwargs["tokenizer"] = tokenizer
else:
    raise RuntimeError("Unsupported SFTTrainer signature. Restore the pinned stack.")
trainer = SFTTrainer(**trainer_kwargs)
# Drop this temporary dictionary so it cannot keep the GPU model alive at reload.
del trainer_kwargs
print("Trainer ready:", type(trainer).__name__, "|", length_key)


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Trainer ready: SFTTrainer | max_length


## 8. Evaluate the base, train, and evaluate the adapter

Baseline evaluation disables adapters on the same quantized base. Both scores
use the identical held-out blocks. Perplexity is `exp(loss)` on their supervised
next-token targets; lower is better. It measures prose prediction, not financial
reasoning or factual accuracy. This validation split is not a separate final test.


In [ ]:
def add_perplexity(metrics, prefix):
    result = dict(metrics)
    loss = float(result[f"{prefix}_loss"])
    if not math.isfinite(loss):
        raise RuntimeError(f"Non-finite {prefix} loss. Inspect data, GPU precision, and learning rate.")
    result[f"{prefix}_perplexity"] = math.exp(loss) if loss < 700 else None
    return result

with model.disable_adapter():
    baseline_metrics = add_perplexity(trainer.evaluate(metric_key_prefix="baseline"), "baseline")
save_json(RUN_DIR / "baseline_metrics.json", baseline_metrics)
print("Base:", baseline_metrics)

train_result = trainer.train()
final_metrics = add_perplexity(trainer.evaluate(metric_key_prefix="eval"), "eval")
comparison = {
    "baseline": baseline_metrics, "adapted": final_metrics,
    "loss_change": final_metrics["eval_loss"] - baseline_metrics["baseline_loss"],
    "training": train_result.metrics,
}
save_json(RUN_DIR / "metrics.json", comparison)
print(json.dumps(comparison, indent=2))


Base: {'baseline_loss': 2.939148426055908, 'baseline_model_preparation_time': 0.0052, 'baseline_runtime': 19.0334, 'baseline_samples_per_second': 0.263, 'baseline_steps_per_second': 0.263, 'baseline_perplexity': 18.899744927138123}


Step,Training Loss,Validation Loss,Model Preparation Time
50,2.433900,2.679068,0.005200


Step,Training Loss,Validation Loss,Model Preparation Time
50,2.433900,2.679068,0.005200


## 9. Save the final adapter, tokenizer, and run record to Drive

The final adapter reflects the last training step. Intermediate checkpoints
contain optimizer/trainer state and are retained separately (latest two).
Do not merge the 4-bit model to create a full precision export here.


In [ ]:
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
save_json(RUN_DIR / "trainer_log.json", trainer.state.log_history)
save_json(ADAPTER_DIR / "run_config.json", config)
installed = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "requirements_freeze.txt").write_text(installed)
for filename in ("adapter_config.json", "adapter_model.safetensors", "tokenizer_config.json"):
    if not (ADAPTER_DIR / filename).is_file():
        raise RuntimeError(f"Expected saved file is missing: {filename}")
print("Saved final adapter:", ADAPTER_DIR)
print("Saved run artifacts:", sorted(p.name for p in RUN_DIR.iterdir()))


## 10. Reload the saved adapter and generate a completion

This verifies the saved files by loading a fresh 4-bit base plus the Drive adapter.
It first releases the training model to avoid holding two copies on the GPU.
After a new Colab session, run installation, imports/GPU checks, and mount Drive,
then set `ADAPTER_PATH` below to the printed adapter folder and run this cell.
The saved configuration supplies the exact base revision.


In [ ]:
ADAPTER_PATH = str(ADAPTER_DIR)  # After a restart, replace with "/content/drive/MyDrive/.../adapter".
for variable in ("trainer", "model", "inference_model", "inference_base"):
    globals().pop(variable, None)
gc.collect()
torch.cuda.empty_cache()

saved_config = json.loads((Path(ADAPTER_PATH) / "run_config.json").read_text())
inference_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_PATH)
if inference_tokenizer.pad_token_id is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token
inference_tokenizer.padding_side = "left"
inference_quant = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
inference_base = AutoModelForCausalLM.from_pretrained(
    saved_config["MODEL_ID"], revision=saved_config["model_sha"],
    quantization_config=inference_quant, torch_dtype=COMPUTE_DTYPE,
    device_map={"": 0}, attn_implementation="sdpa",
)
inference_model = PeftModel.from_pretrained(inference_base, ADAPTER_PATH, is_trainable=False)
inference_model.eval()
inference_model.config.use_cache = True

prompt = "The company's operating cash flow increased during the quarter, primarily because"
inputs = inference_tokenizer(prompt, return_tensors="pt").to("cuda:0")
with torch.inference_mode():
    generated = inference_model.generate(
        **inputs, max_new_tokens=150, do_sample=False, repetition_penalty=1.05,
        eos_token_id=inference_tokenizer.eos_token_id,
        pad_token_id=inference_tokenizer.pad_token_id, use_cache=True,
    )
print(inference_tokenizer.decode(generated[0], skip_special_tokens=True))


The company's operating cash flow increased during the quarter, primarily because of higher sales and lower inventory levels.


## Practical notes and sources

- **Scaling:** increase `SAMPLE_DOCS`, `MAX_SCANNED_ROWS`, and the shuffle buffer for broader
  coverage; increase steps deliberately using the printed approximate pass count.
  A full-corpus run needs a different budget than these Colab defaults.
- **Interrupted training:** checkpoints live in the run's `checkpoints` folder. To resume,
  reconstruct the same data, immutable revisions, model, and trainer using the saved
  configuration, then call `trainer.train(resume_from_checkpoint="/content/drive/MyDrive/.../checkpoints/checkpoint-50")`
  instead of `trainer.train()`. Do not point to `adapter/`: it has no optimizer state.
  Automated cross-session resume is intentionally not assumed.
- **Troubleshooting:** if installation changed already imported packages, restart the session.
  For CUDA/library mismatches start a fresh supported Colab GPU runtime and reinstall;
  for OOM lower `BLOCK_SIZE`. Drive checkpoint writes can pause the progress bar.
- **Data terms:** the corpus combines upstream sources under mixed terms. Review its
  dataset card before redistribution. Nothing in this notebook uploads to the Hub.

Primary references checked while creating this notebook:
[dataset/schema and terms](https://huggingface.co/datasets/alwaysgood/financial-english-source-corpus),
[base model](https://huggingface.co/Qwen/Qwen2.5-1.5B),
[TRL 0.17 SFTTrainer](https://huggingface.co/docs/trl/v0.17.0/sft_trainer),
[PEFT quantization](https://huggingface.co/docs/peft/developer_guides/quantization),
[Transformers 4.51.3 Trainer](https://huggingface.co/docs/transformers/v4.51.3/main_classes/trainer).

**Validation:** notebook structure and Python syntax plus synthetic schema, deduplication,
packing, split, and label-masking checks were validated locally. The full GPU training
run was not executed during notebook creation; it runs in your Colab GPU session.
